# Bus RavKav — AM-Peak Trips by TAZ

Processes the four RavKav bus-trip files (`Input/BusRavKav/`, Tuesdays 2022-05-03/17/24/31):

1. **Stop → TAZ tagging**: every unique bus stop (by `stop_code`, WGS84 lat/lon) is spatially joined to the `TAZ_North` polygons (781 zones, reprojected to the shapefile's Israeli TM CRS).
2. **Filters**: `weekday = 3`; trip hour from the timestamp embedded in `passanger_trip_id` (the `date` column carries no time) ∈ {6, 7, 8}.
3. **Aggregation**, weighted by `total_boardings` (passengers per record), averaged over the four days:
   - OD matrix by TAZ from the journey **origin → destination** stops (`orig_stop` / `dest_stop`);
   - per-TAZ **total boardings and alightings** from the `board_stop` / `alight_stop` columns (the physical boarding/alighting events — these differ from origin/destination on ~21% of records, where the journey involves a transfer).

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import os

FILES = ['Input/BusRavKav/trips_table_2022-05-03.csv',
         'Input/BusRavKav/trips_table_2022-05-17.csv',
         'Input/BusRavKav/trips_table_2022-05-24.csv',
         'Input/BusRavKav/trips_table_2022-05-31.csv']
USECOLS = ['passanger_trip_id', 'weekday', 'total_boardings',
           'orig_stop_code', 'orig_stop_lat', 'orig_stop_lon',
           'dest_stop_code', 'dest_stop_lat', 'dest_stop_lon',
           'board_stop_code', 'board_stop_lat', 'board_stop_lon',
           'alight_stop_code', 'alight_stop_lat', 'alight_stop_lon']

taz = gpd.read_file('Input/TAZ_North/TAZ_North.shp')[['TAZ_NUMBER', 'geometry']]
print(f"TAZ polygons: {len(taz)}")

TAZ polygons: 781


## 1. Collect unique stops and tag them by TAZ polygon

In [2]:
stops = {}
frames = []
for f in FILES:
    df = pd.read_csv(f, usecols=USECOLS)
    frames.append(df)
    for role in ['orig', 'dest', 'board', 'alight']:
        s = df[[f'{role}_stop_code', f'{role}_stop_lat', f'{role}_stop_lon']].dropna()
        s.columns = ['stop_code', 'lat', 'lon']
        for code_, lat, lon in s.drop_duplicates('stop_code').itertuples(index=False):
            stops.setdefault(code_, (lat, lon))
    print(f"{f.split('/')[-1]}: {len(df):,} records")

stops_df = pd.DataFrame([(k, v[0], v[1]) for k, v in stops.items()], columns=['stop_code', 'lat', 'lon'])
pts = gpd.GeoDataFrame(stops_df, geometry=gpd.points_from_xy(stops_df['lon'], stops_df['lat']), crs='EPSG:4326').to_crs(taz.crs)
tagged = gpd.sjoin(pts, taz, how='left', predicate='within')
tagged = tagged.drop_duplicates('stop_code')  # a point on a shared boundary can match twice
stop_to_taz = tagged.set_index('stop_code')['TAZ_NUMBER']
os.makedirs('Output/bus', exist_ok=True)
tagged[['stop_code', 'lat', 'lon', 'TAZ_NUMBER']].to_csv('Output/bus/bus_stops_taz.csv', index=False, float_format='%.6f')
print(f"unique stops: {len(stops_df):,} | inside a TAZ_North polygon: {stop_to_taz.notna().sum():,} "
      f"({stop_to_taz.notna().mean():.1%}) — the rest are outside the northern study area")

trips_table_2022-05-03.csv: 2,476,772 records


trips_table_2022-05-17.csv: 2,813,312 records


trips_table_2022-05-24.csv: 2,826,013 records


trips_table_2022-05-31.csv: 2,870,481 records


unique stops: 27,186 | inside a TAZ_North polygon: 9,924 (36.5%) — the rest are outside the northern study area


## 2. Filter (weekday 3, 6:00–9:00) and aggregate per day

In [3]:
od_days, board_days, alight_days = [], [], []
for f, df in zip(FILES, frames):
    hour = pd.to_datetime(df['passanger_trip_id'].str.split('_', n=1).str[1], errors='coerce').dt.hour
    m = df[(df['weekday'] == 3) & hour.isin([6, 7, 8])].copy()
    kept = m['total_boardings'].sum()
    m['o_taz'] = m['orig_stop_code'].map(stop_to_taz)
    m['d_taz'] = m['dest_stop_code'].map(stop_to_taz)
    m['b_taz'] = m['board_stop_code'].map(stop_to_taz)
    m['a_taz'] = m['alight_stop_code'].map(stop_to_taz)

    od = m.dropna(subset=['o_taz', 'd_taz']).groupby(['o_taz', 'd_taz'])['total_boardings'].sum()
    od_days.append(od)
    board_days.append(m.dropna(subset=['b_taz']).groupby('b_taz')['total_boardings'].sum())
    alight_days.append(m.dropna(subset=['a_taz']).groupby('a_taz')['total_boardings'].sum())
    print(f"{f.split('/')[-1]}: AM-peak passengers {kept:,.0f} | with orig+dest in TAZ_North: "
          f"{od.sum():,.0f} ({od.sum() / kept:.1%})")

trips_table_2022-05-03.csv: AM-peak passengers 774,907 | with orig+dest in TAZ_North: 131,593 (17.0%)


trips_table_2022-05-17.csv: AM-peak passengers 879,602 | with orig+dest in TAZ_North: 147,736 (16.8%)


trips_table_2022-05-24.csv: AM-peak passengers 867,746 | with orig+dest in TAZ_North: 147,399 (17.0%)


trips_table_2022-05-31.csv: AM-peak passengers 860,985 | with orig+dest in TAZ_North: 146,233 (17.0%)


## 3. Average the four days

In [4]:
od_avg = pd.concat(od_days, axis=1).fillna(0).mean(axis=1)
od_matrix = od_avg.unstack().fillna(0)
od_matrix.index = od_matrix.index.astype(int)
od_matrix.columns = od_matrix.columns.astype(int)
od_matrix.index.name = 'orig_taz'
od_matrix.to_csv('Output/bus/bus_od_taz_avg.csv', float_format='%.6g')

ba = pd.concat([pd.concat(board_days, axis=1).fillna(0).mean(axis=1).rename('avg_boardings'),
                pd.concat(alight_days, axis=1).fillna(0).mean(axis=1).rename('avg_alightings')], axis=1).fillna(0)
ba.index = ba.index.astype(int)
ba.index.name = 'TAZ_NUMBER'
ba.to_csv('Output/bus/bus_boardings_alightings_taz.csv', float_format='%.6g')

print(f"OD matrix (avg Tuesday, 6:00–9:00): {od_matrix.shape}, {od_matrix.values.sum():,.0f} passengers/day")
print(f"boardings/alightings table: {len(ba)} TAZs | avg daily boardings {ba['avg_boardings'].sum():,.0f}, "
      f"alightings {ba['avg_alightings'].sum():,.0f}")
print("\ntop 10 TAZs by average AM-peak boardings:")
print(ba.sort_values('avg_boardings', ascending=False).head(10).round(1).to_string())

OD matrix (avg Tuesday, 6:00–9:00): (722, 711), 143,240 passengers/day
boardings/alightings table: 730 TAZs | avg daily boardings 157,187, alightings 147,272

top 10 TAZs by average AM-peak boardings:
            avg_boardings  avg_alightings
TAZ_NUMBER                               
1219               6536.8          9011.5
1517               3818.8          6795.0
4010               2823.8          1462.0
3907               2655.8           873.8
106                2186.0          1135.0
1311               1827.5          2061.8
4015               1776.8          2252.0
1518               1772.5           987.8
802                1694.5           433.8
3615               1680.2          1297.0


## Notes

- The trip hour comes from the timestamp inside `passanger_trip_id` (journey start); the `date` column has no time component. `bus_trip_hour` (the leg's hour) agrees with it on 91% of records — the difference is legs boarded after a journey started in an earlier hour.
- The OD matrix uses journey `orig`/`dest` stops (linked trips); the boardings/alightings table uses the physical `board`/`alight` stops (unlinked events, so a transfer journey counts at both boarding points). Both are weighted by `total_boardings` and averaged over the four Tuesdays.
- Stops outside the `TAZ_North` polygons (e.g. Tel Aviv ends of intercity routes) have no TAZ; trips are kept in the OD matrix only when both ends fall inside the northern study area.
- A fifth date file (`Input/trips_table_2022-05-10.csv`, 2022-05-10) sits outside the `BusRavKav` directory and is **not** included, per the four-file instruction — it can be added in one line if intended.